# Session 07 practical lab: Classification challenge for top-price opportunity scoring

> Warning: This notebook is a teaching lab. The production baseline notebook remains `05_classification_audi_a3_germany.ipynb`.

This lab runs directly from processed CSV files. It does not require cloud credentials, BigQuery access, or warehouse writes.

Head of Data 101 uses one central idea: **the pipeline is the product**.

In this lab, classification predicts whether a listing belongs to the external marketplace `top-price` category. The model output is `probability_top_price`. The business artifact is a review queue, not an automatic acquisition decision.

Course narrative boundary:

- `price_label` is an observed marketplace label.
- `top_price` is the binary target derived from `price_label`.
- `top_price = 1` when `price_label == "top-price"`; otherwise `top_price = 0`.
- Neither `price_label` nor `price_label_id` nor `top_price` can be used as model input features.
- Using the target, or a direct source of the target, as an input feature is target leakage.
- Regression remains separate and predicts `expected_price_eur`.
- BI later combines actual price, expected-price gap, and top-price probability.

For beginner clarity, the first classifier does not use `actual_price_eur` or `price_eur` as a feature. Price can be discussed later as an advanced feature because it may be too close to the marketplace price-tier logic.

**Conceptual note:** Threshold selection is not probability calibration. In this lab we choose an operating rule for review capacity. We do not formally prove that a score of 0.80 means an empirical 80% frequency.


## 3-hour live session agenda

- 00:00-00:20 - Target definition, leakage, and data contract
- 00:20-00:40 - Label distribution and naive baseline
- 00:40-01:20 - Logistic Regression as interpretable classifier
- 01:20-01:55 - One flexible model comparison
- 01:55-02:30 - Threshold policy and precision/recall trade-off
- 02:30-02:50 - BI-ready review queue
- 02:50-03:00 - Discussion and hand-off


## How to work

This is not a pure coding course. Use AI-assisted coding when useful, but make sure you can explain every result.

For each challenge, read the business reason, complete the small coding task or inspect the instructor-led output, and write a short business interpretation in English.

Look for the labels **Student task**, **Instructor-led demo**, **Discussion pause**, and **Optional extension**. They are there to protect class pacing.


## Challenge 0 - Classification data contract and leakage boundary

**Suggested time:** 20 minutes

**Instructor-led demo**

**Business reason:**  
Before modeling, a data scientist must know exactly which dataset contract is being consumed and where the target comes from.

**Objective:**  
Load the processed CSV, validate required columns, create `listing_id` if needed, create `top_price` from `price_label` if needed, and prepare a modeling dataframe.

**Leakage boundary:**
- `price_label` is the observed marketplace label.
- `top_price` is the target.
- `price_label`, `price_label_id`, and `top_price` cannot be features.

**Discussion pause:**
- Is `top_price` already present or created from `price_label`?
- Why should target definition fail loudly rather than be guessed?


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:,.3f}".format)
plt.style.use("default")


def find_repo_root(start):
    for path in [start] + list(start.parents):
        if (path / ".git").exists() or (path / "config" / "project_config.yaml").exists():
            return path
    return start


def load_project_config(config_path):
    config = {}
    if not config_path.exists():
        return config
    for raw_line in config_path.read_text(encoding="utf-8-sig").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or ":" not in line:
            continue
        key, value = line.split(":", 1)
        key = key.strip()
        value = value.strip()
        if value.startswith(("'", '"')) and value.endswith(("'", '"')):
            value = value[1:-1]
        elif value.lower() in ("true", "false"):
            value = value.lower() == "true"
        else:
            try:
                value = int(value)
            except ValueError:
                try:
                    value = float(value)
                except ValueError:
                    pass
        config[key] = value
    return config


def relative_path(path):
    try:
        return str(path.relative_to(PROJECT_ROOT))
    except ValueError:
        return str(path)


def csv_missing_required_columns(path, required_columns):
    try:
        columns = pd.read_csv(path, nrows=0).columns
    except Exception as exc:
        return required_columns, str(exc)
    missing = [column for column in required_columns if column not in columns]
    return missing, None


def csv_contains_required_columns(path, required_columns):
    missing, error = csv_missing_required_columns(path, required_columns)
    return not missing and error is None


def newest_csv_in_folder(folder):
    if not folder.exists():
        return None
    csv_files = [path for path in folder.glob("*.csv") if path.is_file()]
    if not csv_files:
        return None
    return max(csv_files, key=lambda path: path.stat().st_mtime)


def newest_data_csv_with_required_columns(data_root, required_columns):
    if not data_root.exists():
        return None
    candidates = []
    for path in data_root.rglob("*.csv"):
        if path.is_file() and csv_contains_required_columns(path, required_columns):
            candidates.append(path)
    if not candidates:
        return None
    return max(candidates, key=lambda path: path.stat().st_mtime)


REQUIRED_COLUMNS = ["price_label", "mileage_km", "age_years", "power_hp"]
SESSION_07_FULL_CSV = "autoscout24_listings_processed_audi_a3_germany_20251228_205210.csv"
OPTIONAL_COLUMNS = [
    "listing_id",
    "make",
    "model",
    "brand",
    "fuel_type",
    "listing_country",
    "price_eur",
    "actual_price_eur",
    "registration_year",
    "registration_month",
    "price_outlier_iqr",
    "mileage_outlier_iqr",
    "power_outlier_iqr",
    "logical_issue",
]
LEAKAGE_COLUMNS = ["price_label", "price_label_id", "top_price"]


def select_processed_csv(required_columns, preferred_filename):
    preferred_paths = [
        ("Preferred full classroom sample CSV (local)", processed_folder / preferred_filename),
        ("Repo-visible fallback sample", sample_processed_folder / preferred_filename),
    ]
    rejected = []

    for source_label, candidate in preferred_paths:
        if candidate.exists():
            missing, error = csv_missing_required_columns(candidate, required_columns)
            if not missing and error is None:
                return candidate, source_label
            rejected.append((source_label, candidate, missing, error))

    candidate = newest_csv_in_folder(processed_folder)
    if candidate is not None:
        missing, error = csv_missing_required_columns(candidate, required_columns)
        if not missing and error is None:
            return candidate, "Latest local processed CSV"
        rejected.append(("Latest local processed CSV", candidate, missing, error))

    candidate = newest_data_csv_with_required_columns(PROJECT_ROOT / "data", required_columns)
    if candidate is not None:
        return candidate, "Last-resort scan for any CSV with required columns"

    details = []
    for source_label, path, missing, error in rejected:
        if error:
            details.append(f"{source_label}: {relative_path(path)} could not be read ({error})")
        else:
            details.append(f"{source_label}: {relative_path(path)} missing {missing}")
    raise FileNotFoundError(
        "No processed CSV with the required classification columns was found. "
        "Run Notebook 02 to create data/processed/*.csv, or use data/sample/processed/.\n"
        + "\n".join(details)
    )


def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_preprocessor(numeric_columns, categorical_columns, scale_numeric=True):
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))
    numeric_transformer = Pipeline(numeric_steps)
    transformers = [("numeric", numeric_transformer, numeric_columns)]

    if categorical_columns:
        categorical_transformer = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", make_one_hot_encoder()),
        ])
        transformers.append(("categorical", categorical_transformer, categorical_columns))

    return ColumnTransformer(transformers=transformers)


def safe_roc_auc(y_true, y_score):
    if y_score is None or pd.Series(y_true).nunique() < 2:
        return np.nan
    try:
        return roc_auc_score(y_true, y_score)
    except ValueError:
        return np.nan


def evaluate_classifier(model_name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_score = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None
    return {
        "model_name": model_name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": safe_roc_auc(y_test, y_score),
        "predictions": y_pred,
        "probabilities": y_score,
    }


def plot_confusion_matrix(cm, title):
    fig, ax = plt.subplots(figsize=(4.5, 4))
    image = ax.imshow(cm, cmap="Blues")
    ax.set_title(title)
    ax.set_xticks([0, 1], labels=["not top", "top"])
    ax.set_yticks([0, 1], labels=["not top", "top"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    for row in range(cm.shape[0]):
        for col in range(cm.shape[1]):
            ax.text(col, row, int(cm[row, col]), ha="center", va="center", color="black")
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()


def build_threshold_table(y_true, y_score, thresholds):
    rows = []
    for threshold in thresholds:
        y_pred = (y_score >= threshold).astype(int)
        rows.append({
            "threshold": threshold,
            "selected_count": int(y_pred.sum()),
            "selected_rate": float(y_pred.mean()),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
        })
    return pd.DataFrame(rows)


def get_feature_names(preprocessor):
    try:
        names = preprocessor.get_feature_names_out()
        return [name.replace("numeric__", "").replace("categorical__", "") for name in names]
    except Exception:
        return numeric_features + categorical_features


PROJECT_ROOT = find_repo_root(Path.cwd())
PROJECT_CONFIG = load_project_config(PROJECT_ROOT / "config" / "project_config.yaml")
RANDOM_SEED = int(PROJECT_CONFIG.get("random_state", 42))
np.random.seed(RANDOM_SEED)

make_scope = str(PROJECT_CONFIG.get("make", "Audi")).strip()
model_scope = str(PROJECT_CONFIG.get("model", "A3")).strip()
country_scope = str(PROJECT_CONFIG.get("country", "Germany")).strip()
TAG = f"{make_scope}_{model_scope}_{country_scope}".lower().replace(" ", "_")

processed_folder = PROJECT_ROOT / str(PROJECT_CONFIG.get("processed_data_path", "data/processed"))
sample_processed_folder = PROJECT_ROOT / "data" / "sample" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

selected_csv_path, selected_source = select_processed_csv(REQUIRED_COLUMNS, SESSION_07_FULL_CSV)
df_raw = pd.read_csv(selected_csv_path)
row_count_before = len(df_raw)
missing_required = [column for column in REQUIRED_COLUMNS if column not in df_raw.columns]
if missing_required:
    raise ValueError("Missing required classification columns: " + ", ".join(missing_required))

missing_optional = [column for column in OPTIONAL_COLUMNS if column not in df_raw.columns]
df = df_raw.copy()
created_listing_id = "listing_id" not in df.columns
had_top_price = "top_price" in df.columns

for column in ["mileage_km", "age_years", "power_hp", "price_eur", "actual_price_eur"]:
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

df["price_label"] = df["price_label"].astype("string").str.strip().str.lower()
top_price_from_label = df["price_label"].eq("top-price").astype(int)

if had_top_price:
    top_price_text = df["top_price"].astype("string").str.strip().str.lower()
    top_price_text_map = top_price_text.map({"true": 1, "false": 0, "yes": 1, "no": 0, "1": 1, "0": 0})
    existing_top_price = pd.to_numeric(df["top_price"], errors="coerce")
    existing_top_price = existing_top_price.where(existing_top_price.notna(), top_price_text_map)
    invalid_existing_mask = existing_top_price.notna() & ~existing_top_price.isin([0, 1])
    if invalid_existing_mask.any():
        raise ValueError(
            "Existing top_price contains values other than 0/1 for "
            f"{int(invalid_existing_mask.sum())} rows. Fix the target definition before modeling."
        )
    mismatch_mask = existing_top_price.notna() & existing_top_price.astype("Int64").ne(top_price_from_label)
    if mismatch_mask.any():
        raise ValueError(
            "Existing top_price does not match price_label == 'top-price' for "
            f"{int(mismatch_mask.sum())} rows. Fix the target definition before modeling."
        )
    top_price_status = "present and validated against price_label"
else:
    top_price_status = "created from price_label"

df["top_price"] = top_price_from_label

valid_modeling_rows = (
    df["price_label"].notna()
    & df["mileage_km"].gt(0)
    & df["age_years"].ge(0)
    & df["power_hp"].gt(0)
)
rows_removed = int((~valid_modeling_rows).sum())
df = df.loc[valid_modeling_rows].reset_index(drop=True)
if created_listing_id:
    df.insert(0, "listing_id", np.arange(1, len(df) + 1))
    missing_optional = [column for column in missing_optional if column != "listing_id"]

if df.empty:
    raise ValueError("The selected CSV has no valid modeling rows after filtering.")
if df["top_price"].nunique() < 2:
    raise ValueError("The target top_price has only one class after filtering.")

numeric_features = ["mileage_km", "age_years", "power_hp"]
categorical_features = []
if "fuel_type" in df.columns and df["fuel_type"].nunique(dropna=True) > 1:
    categorical_features.append("fuel_type")
feature_columns = numeric_features + categorical_features
excluded_feature_columns = [column for column in LEAKAGE_COLUMNS + ["actual_price_eur", "price_eur"] if column in df.columns]

print("Selected CSV:", relative_path(selected_csv_path))
print("Source priority:", selected_source)
print("Configured scope:", make_scope, model_scope, country_scope)
print("Rows before filtering:", row_count_before)
print("Rows after filtering:", len(df))
print("Rows removed by required-value filter:", rows_removed)
print("Target definition: top_price = 1 when price_label == 'top-price', otherwise 0")
print("top_price status:", top_price_status)
print("Main feature columns:", feature_columns)
print("Excluded from model features because of leakage or beginner clarity:", excluded_feature_columns)
if missing_optional:
    print("Optional columns not found in this CSV:", ", ".join(missing_optional))

print("\nFirst 5 rows:")
display(df.head())

print("Missing values summary:")
missing_summary = df.isna().sum().sort_values(ascending=False).to_frame("missing_values")
display(missing_summary[missing_summary["missing_values"] > 0])


## Challenge 1 - Label distribution and naive baseline

**Suggested time:** 25 minutes

**Student task with helper functions**

**Business reason:**  
Classification starts with the label. A model cannot be interpreted if the positive class is not understood. A classifier must also beat a simple baseline.

**Objective:**  
Inspect positive-class share and evaluate a majority-class baseline.

**Expected output format:**
- `price_label` frequency table.
- `top_price` count and percentage table.
- Class distribution bar chart.
- Majority-class baseline metrics and confusion matrix.

**Discussion pause:**
- How common is the positive class?
- Why can accuracy be misleading?
- What happens if the review queue misses all positives?


In [ ]:
# Challenge 1 guided scaffold.
# The helper functions are provided; your task is to inspect and interpret the outputs.

# Your code here
# price_label_summary = ...
# target_summary = ...
# display(price_label_summary)
# display(target_summary)
#
# X = df[feature_columns].copy()
# y = df["top_price"].astype(int)
# X_train, X_test, y_train, y_test = train_test_split(
#     X,
#     y,
#     test_size=0.20,
#     random_state=RANDOM_SEED,
#     stratify=y,
# )
#
# majority_class = y_train.mode()[0]
# y_pred_baseline = np.repeat(majority_class, len(y_test))
# baseline_cm = confusion_matrix(y_test, y_pred_baseline, labels=[0, 1])
# plot_confusion_matrix(baseline_cm, "Majority-class baseline")


### Your label and baseline interpretation

1. Positive class share:
2. Majority-class baseline accuracy:
3. Did the baseline detect positives?
4. Which metric makes the baseline failure visible?


## Challenge 2 - Logistic Regression as first classifier

**Suggested time:** 40 minutes

**Student task**

**Business reason:**  
A professional data scientist starts with a transparent model before moving to flexible models.

**Objective:**  
Train Logistic Regression using a clean feature set.

**Main feature set:**
- `mileage_km`
- `age_years`
- `power_hp`
- `fuel_type` if available

**Do not include:** `price_label`, `price_label_id`, `top_price`, `actual_price_eur`, or `price_eur`.

**Expected output format:**
- Metric table.
- Confusion matrix.
- ROC-AUC if available.
- Short interpretation of precision and recall.

**Instructor-led demo:** Coefficients can be shown if time allows, but they are not the main coding task.


In [ ]:
# Challenge 2 scaffold.
# Use the shared preprocessing and evaluation helpers.

# Your code here
# minority_share = y_train.value_counts(normalize=True).min()
# logreg_class_weight = "balanced" if minority_share < 0.30 else None
# logreg_pipeline = Pipeline([
#     ("preprocess", build_preprocessor(numeric_features, categorical_features, scale_numeric=True)),
#     ("model", LogisticRegression(max_iter=2000, class_weight=logreg_class_weight)),
# ])
# logreg_result = evaluate_classifier("Logistic Regression", logreg_pipeline, X_train, X_test, y_train, y_test)
# display(pd.DataFrame([{k: v for k, v in logreg_result.items() if k not in ["predictions", "probabilities"]}]))
# plot_confusion_matrix(confusion_matrix(y_test, logreg_result["predictions"], labels=[0, 1]), "Logistic Regression")


### Your Logistic Regression interpretation

1. Precision:
2. Recall:
3. Is the model useful as a first review filter?
4. What business cost appears if recall is too low?


## Challenge 3 - One flexible model comparison

**Suggested time:** 35 minutes

**Student task**

**Business reason:**  
Flexible models can capture non-linear patterns, but they may be harder to explain.

**Objective:**  
Compare Logistic Regression with one flexible model: Random Forest.

**Expected output format:**
- Comparison table.
- Confusion matrices.
- Optional Random Forest feature importance chart.

**Discussion pause:**
- Which model is better for a review queue?
- Which metric supports that decision?
- Is the gain worth the complexity?


In [ ]:
# Challenge 3 scaffold.
# Compare Logistic Regression with Random Forest only in the core path.

# Your code here
# forest_class_weight = "balanced_subsample" if use_class_weight else None
# rf_pipeline = Pipeline([
#     ("preprocess", build_preprocessor(numeric_features, categorical_features, scale_numeric=False)),
#     ("model", RandomForestClassifier(
#         n_estimators=100,
#         max_depth=8,
#         random_state=RANDOM_SEED,
#         n_jobs=-1,
#         class_weight=forest_class_weight,
#     )),
# ])
# rf_result = evaluate_classifier("Random Forest", rf_pipeline, X_train, X_test, y_train, y_test)
# Compare logreg_result and rf_result using precision, recall, F1, and ROC-AUC.


### Your model comparison decision

1. Selected model for the review queue:
2. Metric evidence:
3. Trade-off between performance and explainability:


## Challenge 4 - Threshold policy and review capacity

**Suggested time:** 35 minutes

**Student task and discussion pause**

**Business reason:**  
A classifier outputs probabilities. The threshold is an operational decision, not a model truth.

**Objective:**  
Analyze how different probability thresholds change precision, recall, F1, and number of listings selected.

**Expected output format:**
- Threshold table for 0.30, 0.40, 0.50, 0.60, 0.70.
- Plot of threshold vs precision and recall.
- Recommended threshold or Top-N policy.

**Discussion pause:**
- False positives consume analyst time.
- False negatives miss opportunities.
- 0.50 is not automatically correct.
- Threshold is an operating policy for review capacity.


In [ ]:
# Challenge 4 scaffold.
# Select the model you would use for a review queue, then test policy thresholds.

# Your code here
# selected_model_name = "..."
# selected_result = model_results[selected_model_name]
# threshold_analysis_df = build_threshold_table(
#     y_test,
#     selected_result["probabilities"],
#     thresholds=[0.30, 0.40, 0.50, 0.60, 0.70],
# )
# display(threshold_analysis_df)
# Plot threshold vs precision and recall.


### Your threshold policy recommendation

1. Recommended threshold or Top-N rule:
2. Expected review volume:
3. Main business trade-off:
4. Why 0.50 is not automatically correct:


## Challenge 5 - BI-ready classification review queue

**Suggested time:** 20 minutes

**Instructor-led demo with student interpretation**

**Business reason:**  
A classification model becomes useful only when it creates an actionable review artifact.

**Objective:**  
Fit the selected model on all valid rows, generate `probability_top_price`, and create `classification_review_queue`.

**Expected output format:**
- `probability_top_price`
- `predicted_top_price`
- `classification_review_queue`
- CSV outputs saved under `reports/`

**Important operational boundary:**  
In historical evaluation, it is fine to keep `price_label` and `top_price` in the review queue for validation. In real future scoring, `price_label` and `top_price` would not be known and should not be relied on operationally.

**Discussion pause:**
- Which listings deserve analyst review first?
- Which predictions might be model artifacts?
- How should Power BI display this output?


In [ ]:
# Challenge 5 scaffold.
# The instructor can run this after the threshold discussion.

# Your code here
# final_model = ...
# final_model.fit(X, y)
# probability_top_price = final_model.predict_proba(X)[:, 1]
# predicted_top_price = (probability_top_price >= recommended_threshold).astype(int)
#
# Build classification_review_queue with context columns.
# Save:
# - reports/{TAG}_classification_review_queue.csv
# - reports/{TAG}_classification_model_comparison.csv
# - reports/{TAG}_classification_threshold_analysis.csv


### Your BI hand-off conclusion

1. Three listings for analyst review:
2. One model risk:
3. One missing business input before a real decision:
4. What should Power BI show?


## Optional extension - Probability calibration

**Optional extension**

Calibration asks whether predicted probabilities match observed frequencies. For example, among listings scored near 0.80, do roughly 80% actually belong to the positive class?

This is different from threshold selection. Threshold selection chooses an operating rule for review capacity. Calibration evaluates whether the probability scale itself is trustworthy.

Do not add calibration to the core path. If used, discuss it conceptually or demonstrate `CalibratedClassifierCV` after class.


## Final discussion: what did we learn about classification as a prioritization layer?

Use these prompts for the final class discussion:

- Why did we inspect the target before training?
- Why did we build a naive baseline?
- Why is accuracy dangerous in imbalanced classification?
- Why is `price_label` allowed as the source of the target but not as a feature?
- What is leakage?
- How do precision and recall map to business costs?
- Why is threshold selection a policy decision?
- How does `probability_top_price` become a BI-ready prioritization layer?

Final message:

**A useful classifier does not end in a metric. It ends in a probability layer that helps people prioritize review, understand trade-offs, and make better decisions.**
